# Exploratory Data Analysis: Amazon US Customer Reviews

Quick look at one TSV file before training the Spark models.
Run `python scripts/download_data.py` first, or point `DATA_FILE` at any local TSV.

In [ ]:
import csv
import glob

import pandas as pd
import plotly.express as px

DATA_FILE = (glob.glob("../data/raw/*.tsv") or glob.glob("data/raw/*.tsv"))[0]
print("Using:", DATA_FILE)

## Load the data

In [ ]:
df = pd.read_csv(
    DATA_FILE,
    sep="\t",
    quoting=csv.QUOTE_NONE,   # review text contains stray quote characters
    on_bad_lines="skip",
    nrows=200_000,            # keep the notebook fast
)
print(df.shape)
df.head()

## Schema and missing values

In [ ]:
df.info()
df.isna().sum().sort_values(ascending=False).head(10)

## Star rating distribution

In [ ]:
ratings = df["star_rating"].value_counts().sort_index().reset_index()
ratings.columns = ["star_rating", "reviews"]
px.bar(ratings, x="star_rating", y="reviews", title="Reviews per star rating")

## Sentiment labels

Same rule as the training script: 4-5 stars = positive, 1-2 = negative, 3 = neutral.

In [ ]:
def to_sentiment(stars):
    if stars >= 4:
        return "positive"
    if stars <= 2:
        return "negative"
    return "neutral"

df["sentiment"] = df["star_rating"].apply(to_sentiment)
df["sentiment"].value_counts(normalize=True).round(3)

## Review length

In [ ]:
df["review_length"] = df["review_body"].fillna("").str.len()
df["word_count"] = df["review_body"].fillna("").str.split().str.len()
px.histogram(df[df["review_length"] < 2000], x="review_length", color="sentiment",
             nbins=60, title="Review length (characters) by sentiment")

In [ ]:
df.groupby("sentiment")[["review_length", "word_count"]].mean().round(1)

## Verified purchases and helpfulness

In [ ]:
print(df["verified_purchase"].value_counts(normalize=True).round(3))
df.groupby("sentiment")[["helpful_votes", "total_votes"]].mean().round(2)

## Takeaways

- _Write what you notice here (class balance, length differences, missing data)._
- _These observations motivate the class-weighting option and the length/word-count features in the model._